In [1]:

import requests
import pandas as pd
import numpy as np
import re
import base64
import json
import time

BASE_URL = "https://polls.iplt20.com/widget/welcome/get_data"

def fetch_x_api_key():
    url = 'https://polls.iplt20.com/bundle.js?v=1.4'
    headers = {
        'sec-ch-ua-platform': '"Linux"',
        'Referer': 'https://polls.iplt20.com/?entity_matchId=87747&matchId=13390407062092&ipl=1',
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
        'sec-ch-ua': '"Brave";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
        'sec-ch-ua-mobile': '?0',
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=100)
        response.raise_for_status()
        # Extract token from the response using regex
        match = re.search(r'"access_token\\":\\"([^"]+)\\"', response.text)
        if match:
            return match.group(1)
        else:
            print("x-api-key not found in response.")
            return None
    except requests.RequestException as e:
        print(f"Request failed: {e}")
        return None
    

def fetch_x_token_key(x_api_key):
    url = f'https://polls.iplt20.com/widget/welcome/get_data?path=matches/87747/innings/info&token=66'
    headers = {
        'accept': '*/*',
        'accept-language': 'en-GB,en;q=0.9',
        'priority': 'u=0, i',
        'referer': 'https://polls.iplt20.com/?entity_matchId=87747&matchId=13390407062092&ipl=1',
        'sec-ch-ua': '"Brave";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"Linux"',
        'sec-fetch-dest': 'empty',
        'sec-fetch-mode': 'cors',
        'sec-fetch-site': 'same-origin',
        'sec-gpc': '1',
        'user-agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
        'x-api-key': x_api_key,
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=100)
        response.raise_for_status()
        # Extract the token key from the response
        match = re.search(r'"key":"([^"]+)"', response.text)
        if match:
            return match.group(1)
        else:
            print("x-token-key not found in response.")
            return None
    except requests.RequestException as e:
        print(f"Request failed: {e}")
        return None
    

def fetch_payload(x_api_key, x_token_key, req_url):
    # url = 'https://polls.iplt20.com/widget/welcome/get_data?path=Delivery_1_2_4_13390407062092.json'
    url = req_url
    headers = {
        'accept': '*/*',
        'accept-language': 'en-GB,en;q=0.8',
        'priority': 'u=0, i',
        'referer': 'https://polls.iplt20.com/?entity_matchId=87748&matchId=13390493628543&ipl=1',
        'sec-ch-ua': '"Brave";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"Linux"',
        'sec-fetch-dest': 'empty',
        'sec-fetch-mode': 'cors',
        'sec-fetch-site': 'same-origin',
        'sec-gpc': '1',
        'user-agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
        'x-api-key': x_api_key,
        'x-token-key': x_token_key,
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=100)
        response.raise_for_status()
        match = re.search(r'"payload":"([^"]+)"', response.text)
        if match:
            return match.group(1)
        else:
            print("Payload not found in response.")
            return None
    except requests.RequestException as e:
        print(f"Request failed: {e}")
        return None
    
def decode_payload(payload, key):
    # Decode from base64
    encrypted = base64.b64decode(payload)
    # XOR decryption
    decrypted = ''.join(
        chr(b ^ ord(key[i % len(key)])) for i, b in enumerate(encrypted)
    )
    # Parse JSON
    return json.loads(decrypted)

def fetch_bbb_data(inning, over, ball, hawkID, rate_limit=0):

    url = f"{BASE_URL}?path=Delivery_{inning}_{over}_{ball}_{hawkID}.json"
    
    x_api_key = fetch_x_api_key()
    if not x_api_key:
        print("Failed to retrieve x-api-key.")
        return None

    x_token_key = fetch_x_token_key(x_api_key)
    if not x_token_key:
        print("Failed to retrieve x-token-key.")
        return None
        
    payload = fetch_payload(x_api_key, x_token_key, url)
    if not payload and rate_limit<2:
        time.sleep(1)
        fetch_bbb_data(inning, over, ball, hawkID, rate_limit=0)
        rate_limit += 1

    if not payload:
        print("Failed to retrieve payload.")
        return None
        
    # Decode the payload
    key = "ran_js_my_tok"
    try:
        decoded = decode_payload(payload, key)

        return decoded
        # print("SUCCESS Decoded Data:")
        # print(json.dumps(decoded, indent=2))
    except Exception as e:
        print("FAILED to decode:", str(e))
        return None

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# List of (inning, over, ball, hawkID) calls you want to make
tasks = [(1, i, j, 13391098419133) for i in range(10) for j in range(1, 8)]

results = []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(fetch_bbb_data, *t) for t in tasks]
    for future in as_completed(futures):
        print(future.result())

# print(f"Fetched {len(results)} deliveries")


In [11]:
fetch_bbb_data(1, 6, 5, 13391098419133)

{'country': 'INDIA',
 'format': 'T20',
 'international': None,
 'match': {'battingTeam': {'batsman': {'id': '13388586128003',
    'isRightHanded': True,
    'name': 'SUNIL NARINE'},
   'batsmanPartner': {'id': '12865946879656',
    'isRightHanded': True,
    'name': 'Ajinkya Rahane'},
   'home': True,
   'id': '12865944342828',
   'name': 'KOLKATA-KNIGHT-RIDERS'},
  'bowlingTeam': {'bowler': {'id': '13359627957293',
    'isRightHanded': True,
    'name': 'ANSHUL KAMBOJ',
    'spell': 0},
   'bowlerPartner': {'id': None, 'isRightHanded': None, 'name': None},
   'home': False,
   'name': 'CHENNAI-SUPER-KINGS'},
  'delivery': {'additionalEventInformation': {'dropped': None},
   'deliveryNumber': {'ball': 5, 'day': 1, 'innings': 1, 'over': 6},
   'deliveryType': 'Seam',
   'fielderPosition': {'1st Slip': False,
    '2nd Slip': False,
    '3rd Slip': False,
    '4th Slip': False,
    '5th Slip': False,
    'Backwards Point': False,
    'Cover': False,
    'Cow Corner': False,
    'Deep Back

In [1]:
from pathlib import Path
import pandas as pd

folder = Path('bcci_hawkeye_data')

df = pd.concat(
    [pd.read_csv(file) for file in folder.glob('*.csv')],
    ignore_index=True
)

In [2]:
df = df[(df['MatchID'] >= 1896) & (df['release_speed'].isna())]
len(df)

4

In [3]:
df_short = df[['InningsNo', 'OverNo', 'BallCount', 'MatchID']]

In [4]:
df_short.head()

,InningsNo,OverNo,BallCount,MatchID
20701,2,21,1,1896
21339,4,10,4,1896
28774,1,43,6,1897
30196,3,44,2,1897


In [5]:
match_hawk = {}

with open('./bcci_shot_data/Men/hawkeyeid_matchid.csv', 'r') as f:
        next(f)
        for l in f:
            hawk_match_pair = l.replace(' ', '').strip().split(',')

            if int(hawk_match_pair[0]) >= 1896:
                  match_hawk[int(hawk_match_pair[0])]  = int(hawk_match_pair[1])
                # hawkeye_main('Test', hawk_match_pair[0], hawk_match_pair[1], 'ipl')

In [6]:
from data_scrapper import *
from process_data import *
import pandas as pd
import math

In [ ]:
import math
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

FIELDS = [
    'MatchID', 'InningsNo', 'BattingTeamID', 'TeamName', 'BatsManName', 'BowlerName', 'BowlerType', 'OverNo',
    'BallNo', 'Runs', 'BallRuns', 'ActualRuns', 'IsOne', 'IsTwo', 'IsThree', 'IsDotball', 'Extras', 'IsWide', 
    'IsNoBall', 'IsBye', 'IsLegBye', 'IsFour', 'IsSix', 'IsWicket', 'WicketType', 'Wickets', 'IsBowlerWicket', 
    'BallName', 'Day', 'SESSION_NO', 'IsExtra', 'SNO', 'Xpitch', 'Ypitch', 'RunRuns', 'IsMaiden', 'OverImage', 
    'BowlTypeID', 'BowlTypeName', 'ShotTypeID', 'ShotType', 'IsBouncer', 'IsFreeHit', 'BallCount', 'BCCheck', 
    'TotalRuns', 'TotalWickets', 'BOWLING_LINE_ID', 'BOWLING_LENGTH_ID', 'FiveHaul', 'Flag', 'FlagSet', 'PenaltyRuns', 
    'IsFifty', 'IsHundred', 'IsTwoHundred', 'IsHattrick', 'release_speed', 'initial_angle', 'release_x', 'release_y', 
    'release_z', 'pre_bounce_ax', 'pre_bounce_ay', 'pre_bounce_az', 'pre_bounce_vx', 'pre_bounce_vy', 'pre_bounce_vz', 
    'bounce_angle', 'cof', 'cor', 'pbr', 'shot_attacked', 'shot_played', 'shot_info', 'crease_reaction_time', 
    'interception_reaction_time', 'bounce_x', 'bounce_y', 'post_bounce_ax', 'post_bounce_ay', 'post_bounce_az', 
    'post_bounce_vx', 'post_bounce_vy', 'post_bounce_vz', 'impact_x', 'impact_y', 'impact_z', 'crease_x', 'crease_y', 
    'crease_z', 'drop_angle', 'stump_x', 'stump_y', 'stump_z', 'swing', 'deviation', 'swing_dist', 'six_dist', 'ground', 
    'date', 'season'
]

def process_row(i):
    data = fetch_bbb_data(i[0], i[1], i[2], match_hawk[i[3]])
    ball_data = {key: np.nan for key in FIELDS}
    
    ball_data_checks = df.loc[
        (df['MatchID'] == i[3]) & 
        (df['OverNo'] == i[1]) & 
        (df['BallCount'] == i[2]) & 
        (df['InningsNo'] == i[0])
    ]
    fill_non_hawkeye_data(ball_data, ball_data_checks)
    processData(ball_data, data)
    
    if not math.isnan(ball_data['release_speed']):
        return ball_data
    return None


total_data = []
with ThreadPoolExecutor(max_workers=10) as executor:  # adjust workers based on system
    futures = [executor.submit(process_row, i) for i in df_short.values]
    
    for future in as_completed(futures):
        result = future.result()
        if result is not None:
            total_data.append(result)

if total_data:
    pd.DataFrame(total_data).to_csv('test.csv', index=False)
